In [1]:
!pip install psycopg2-binary sqlalchemy pandas
!pip install google-generativeai
!pip install python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import psycopg2
import re
import glob
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import google.generativeai as genai
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

C:\Users\dhrub\Mayo-CoRAL\venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_KEY"))

model = genai.GenerativeModel('gemini-2.0-flash')


In [4]:
!pip install -q google-generativeai python-dotenv pandas openpyxl



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
Pipeline 1 - Single run with filters and columns (ONE-STAGE PROMPT).
"""

import os, time, random, json, ast
import re
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN  = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT = "runs/query-chosen-filters-columns_with_runs_pipeline1.xlsx"
SHEET_NAME     = 0
QUESTION_COLUMN = "question"   # auto-detected if missing
RUNS_PER_QUESTION = 5          # adjust as needed
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MODEL_NAME = "gemini-2.0-flash"

# Definition files (pointed to your uploaded assets)
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"  # filter CATEGORY NAMES only
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"        # column definitions
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"        # full filter (categories + values)

# Columns that are always included by default (do NOT count toward the limit below)
REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6  # number of columns the model may add on top of REQUIRED_COLS

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

generation_config_json = {
    "response_mime_type": "application/json"
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()


def extract_json(text: str):
    raw = strip_code_fences(text or "")
    try:
        return json.loads(raw)
    except Exception:
        pass
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")


def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.2
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=generation_config_json,
                request_options={"timeout": 90},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.25)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)

# ============================
# SINGLE-STAGE PROMPT BUILDER
# ============================

def build_single_stage_prompt(question: str) -> str:
    """
    Build ONE prompt that includes:
      - Filter CATEGORY NAMES
      - Full FILTER definitions (categories + values, including 'All')
      - COLUMN definitions
    Ask Gemini to return the FINAL merged JSON in ONE response:

    {
      "selected_filter": {"Filter Category 1": "Chosen Value", ...},   # omit any 'All'
      "selected_column": { "Column 1":"NCT", "Column 2":"PMID", ... }  # enumerated object
    }
    """
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  },\n'
        '  "selected_column": {\n'
        '    "Column 1": "Name",\n'
        '    "Column 2": "Name",\n'
        '    "Column 3": "Name",\n'
        '    "Column 4": "Name",\n'
        '    "Column 5": "Name",\n'
        '    "Column 6": "Name",\n'
        '    "Column 7": "Name",\n'
        '    "Column 8": "Name",\n'
        '    "Column 9": "Name",\n'
        '    "Column 10": "Name"\n'
        '  }\n'
        '}'
    )

    example = (
        '{'
        '"selected_filter":{"Cancer Type":"NSCLC","Trial Phase":"Phase 3","Type of Therapy":"Combination Therapy"},'
        '"selected_column":{"Column 1":"NCT","Column 2":"PMID","Column 3":"Authors","Column 4":"Year","Column 5":"Primary Endpoint","Column 6":"Sample Size","Column 7":"Name of ICI","Column 8":"Control regimen","Column 9":"Monotherapy/combination","Column 10":"Lines of treatment"}'
        '}'
    )

    # Clear, strict, one-shot instructions
    return (
        "You are a medical expert researching cancer trials. Build the ENTIRE selection in ONE step.\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Strict rules:\n"
        f"- Columns: ALWAYS include these first (do NOT count toward the limit): {', '.join(REQUIRED_COLS)}.\n"
        f"- You may add at most {MAX_ADDITIONAL_COLS} additional columns beyond those required.\n"
        "- The enumerated object keys MUST be exactly 'Column 1', 'Column 2', ... in display order.\n"
        "- Use *exact* names from the available column list.\n"
        "- Filters: choose only categories/values justified by the question. If a category would be 'All', OMIT that category entirely.\n"
        "- Use *exact* category and value strings from the available filter definitions.\n"
        "- Use double quotes everywhere. No trailing commas. No explanations.\n\n"
        "Available filter CATEGORY NAMES (for reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available FULL filter categories and values (use exact strings; omit categories that would be 'All'):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n\n"
        "Return JSON now. Example of valid formatting (not prescriptive):\n"
        f"{example}"
    )

# ============================
# NORMALIZATION & CONVERSION
# ============================

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    """Accepts a mapping like {'Column 1': 'NCT', ...} and returns values ordered by numeric index."""
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val.strip()]


def normalize_selected_column_object(selected_column) -> dict:
    """Ensure REQUIRED_COLS are present first, dedupe, and rebuild enumerated mapping with cap."""
    # Convert input (list or dict) → ordered list
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # Ensure required at the front, keep order & dedupe
    out_list, seen = [], set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc); seen.add(rc)
    for c in cols:
        if c not in seen:
            seen.add(c); out_list.append(c)

    # Keep only up to REQUIRED + MAX_ADDITIONAL_COLS
    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]

    # Build enumerated object
    return {f"Column {i+1}": name for i, name in enumerate(out_list)}


def normalize_selected_filter(selected_filter: dict) -> dict:
    """Drop empty/None/'All' values and trim keys/values."""
    out = {}
    for k, v in (selected_filter or {}).items():
        if v is None:
            continue
        s = str(v).strip()
        if not s or s.lower() == "all":
            continue
        out[str(k).strip()] = s
    return out

# ============================
# CORE RUNNERS
# ============================

def run_once_for_question(question: str, seed: int | None = None) -> tuple[dict, float]:
    """Single run with timing measurement."""
    prompt = build_single_stage_prompt(question)
    start = time.time()
    text = call_model_with_retries(prompt, seed=seed)
    elapsed = time.time() - start

    try:
        obj = extract_json(text)
    except Exception:
        obj = {}

    selected_filter_raw  = obj.get("selected_filter", {}) if isinstance(obj, dict) else {}
    selected_column_raw  = obj.get("selected_column", {}) if isinstance(obj, dict) else {}

    selected_filter = normalize_selected_filter(selected_filter_raw)
    selected_column_obj = normalize_selected_column_object(selected_column_raw)

    merged = {
        "selected_filter": selected_filter,
        "selected_column": selected_column_obj,
    }
    return merged, elapsed


def run_n_times_for_question(question: str, n_runs: int):
    outputs, timings = [], []
    for _ in range(n_runs):
        seed = random.randint(1, 10_000_000)
        merged, elapsed = run_once_for_question(question, seed=seed)
        outputs.append(json.dumps(merged, ensure_ascii=False))
        timings.append(elapsed)
        time.sleep(DELAY_BETWEEN_CALLS_SEC)
    return outputs, timings

# ============================
# MAIN
# ============================

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question", "query", "prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(
                f"Couldn't find a question column named '{QUESTION_COLUMN}'. Available columns: {list(df.columns)}"
            )

    run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
    for c in run_cols:
        if c not in df.columns:
            df[c] = ""

    # Collect timings in parallel DataFrame
    timings_records = []

    for idx, row in df.iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower() == "nan":
            continue
        print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
        merged_runs, timings = run_n_times_for_question(q, RUNS_PER_QUESTION)
        for i, merged_json in enumerate(merged_runs):
            df.at[idx, run_cols[i]] = merged_json
        timings_records.append({
            "row_index": idx,
            "question": q,
            **{f"run_{i+1}_time_sec": t for i, t in enumerate(timings)}
        })

    # Save both sheets
    timings_df = pd.DataFrame(timings_records)
    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT}")


if __name__ == "__main__":
    main()



Processing row 0: Show me the treatment regimen and names of immune checkpoint inhibitors of the c...
Processing row 1: List the names and clinical setting of all combination ICIs given in melanoma pa...
Processing row 2: What are the primary endpoints and sample sizes in trials assessing CTLA4 agents...
Processing row 3: Get all phase 3 studies assessing monotherapy for urothelial/bladder cancer and ...
Processing row 4: Which trials included patients after surgery in NSCLC?
Processing row 5: Provide treatment and comparator names of trials comparing CTLA4 and PD1 agents ...
Processing row 6: Provide all information of studies assessing PD-L1 inhibitors.
Processing row 7: Provide trial names of all the studies where the sample size is equal to or grea...
Processing row 8: Provide treatment names for all the trials where the primary endpoint is overall...
Processing row 9: Provide trial names of all the studies after 2020


In [55]:
"""
Pipeline 2 - Filter names + column names on first stage before filter selection based on names
"""


import os, time, random, json, ast
import re
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN  = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT = "runs/query-chosen-filters-columns_with_runs_pipeline2.xlsx"
SHEET_NAME     = 0
QUESTION_COLUMN = "question"  # auto-detected if missing
RUNS_PER_QUESTION = 5          # adjust as needed
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MODEL_NAME = "gemini-2.0-flash"

# Definition files
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"  # filter CATEGORY NAMES only
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"        # column definitions
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"       # full filter (categories + values)

# Columns that are always included by default (do NOT count toward the limit below)
REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6  # number of columns the model may add on top of REQUIRED_COLS

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

generation_config_json = {
    "response_mime_type": "application/json"
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()


def extract_json(text: str):
    raw = strip_code_fences(text or "")
    try:
        return json.loads(raw)
    except Exception:
        pass
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")


def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.2
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=generation_config_json,
                request_options={"timeout": 90},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.25)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)


# ============================
# PROMPT BUILDERS (2-STAGE)
# ============================

def build_stage1_prompt(question: str) -> str:
    """
    Stage 1: One prompt that includes COLUMN definitions and FILTER NAMES (categories only).
    Ask for: (a) which filter category NAMES are relevant; (b) up to MAX_ADDITIONAL_COLS columns.

    Return JSON ONLY in this shape (selected_column is an *object* with enumerated keys):
    {
      "selected_filter_names": ["Filter Category 1", "Filter Category 2"],
      "selected_column": {
        "Column 1": "Name",
        "Column 2": "Name",
        "Column 3": "Name",
        "Column 4": "Name",
        "Column 5": "Name",
        "Column 6": "Name"
      }
    }
    """
    schema = (
        '{\n'
        '  "selected_filter_names": ["Filter Category 1", "Filter Category 2"],\n'
        '  "selected_column": {\n'
        '    "Column 1": "Name",\n'
        '    "Column 2": "Name",\n'
        '    "Column 3": "Name",\n'
        '    "Column 4": "Name",\n'
        '    "Column 5": "Name",\n'
        '    "Column 6": "Name"\n'
        '  }\n'
        '}'
    )

    example = (
        '{"selected_filter_names":["Cancer type","Trial phase","Class of ICI"],'
        '"selected_column":{"Column 1":"NCT","Column 2":"PMID","Column 3":"Authors","Column 4":"Year","Column 5":"Primary Endpoint(s)","Column 6":"Sample Size"}}'
    )

    return (
        # Column-prompt rules (like original), extended to also return filter names
        "You are a medical expert researching cancer. You also have a set of available columns with definitions. "
        f"You must choose up to {MAX_ADDITIONAL_COLS} relevant columns for the question, but NCT, PMID, Authors, and Year are ALWAYS included by default unless explicitly excluded.\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Rules:\n"
        "- Use double quotes.\n"
        "- Ensure NCT, PMID, Authors, and Year are present among the chosen columns, unless the question explicitly says otherwise.\n"
        f"- Choose at most {MAX_ADDITIONAL_COLS} additional columns beyond the four default columns.\n"
        "- Include only columns justified by the question.\n"
        "- Also select the relevant FILTER CATEGORY NAMES (do NOT assign values yet).\n"
        "- Use exact names from the lists provided.\n"
        "- Do not include any explanation.\n\n"
        "Available filter CATEGORY NAMES (choose names only; do NOT assign values):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n\n"
        "Return JSON now. Example of valid formatting (not prescriptive):\n"
        f"{example}"
    )



def build_stage2_prompt(question: str, selected_filter_names: list[str]) -> str:
    """
    Stage 2: Based on selected filter NAMES, pick specific values using the full filter definitions.
    Follow the original filter-prompt rules: omit categories where the correct value is 'All'.

    Return JSON ONLY:
    {
      "selected_filter": {"Filter Category 1": "Chosen Value", "Filter Category 2": "Chosen Value"}
    }
    """
    filters_json = json.dumps({"selected_filter_names": selected_filter_names}, ensure_ascii=False)
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  }\n'
        '}'
    )

    return (
        # Filter-prompt rules (like original)
        "You are a medical expert researching cancer.\n"
        "You have an interface where you may select zero or more filters. Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Rules:\n"
        "- Omit any filters whose value would be 'All'.\n"
        "- Include only filters that are justified by the question.\n"
        "- Use double quotes.\n"
        "- Do not include any explanation.\n\n"
        f"You may ONLY choose values for these filter category names: {filters_json}.\n\n"
        "Available filter categories and definitions (use exact category and value strings):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now:"
    )


# ============================
# NORMALIZATION & CONVERSION
# ============================

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    """Accepts a mapping like {"Column 1": "NCT", ...} and returns values ordered by numeric index."""
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val.strip()]


def normalize_selected_column_object(selected_column) -> dict:
    """Ensure REQUIRED_COLS are present first, dedupe, and rebuild enumerated mapping."""
    # Convert input (list or dict) → ordered list
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # Ensure required at the front, keep order & dedupe
    out_list = []
    seen = set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc)
            seen.add(rc)
    for c in cols:
        if c not in seen:
            seen.add(c)
            out_list.append(c)

    # Keep only up to REQUIRED + MAX_ADDITIONAL_COLS
    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]

    # Build enumerated object
    out_obj = {f"Column {i+1}": name for i, name in enumerate(out_list)}
    return out_obj


def normalize_filter_names(names: list[str]) -> list[str]:
    names = [n for n in (names or []) if isinstance(n, str) and n.strip()]
    seen = set()
    unique = []
    for n in names:
        if n not in seen:
            seen.add(n)
            unique.append(n)
    return unique[:10]




def normalize_selected_filter(selected_filter: dict) -> dict:
    # Drop empty/None/"All" values
    out = {}
    for k, v in (selected_filter or {}).items():
        if v is None:
            continue
        s = str(v).strip()
        if not s or s.lower() == "all":
            continue
        out[str(k).strip()] = s
    return out


# ============================
# CORE RUNNERS
# ============================

def canonicalize_names(names):
    out = []
    for n in names:
        nn = FILTER_NAME_CANON.get(n.strip(), n.strip())
        if nn not in out:
            out.append(nn)
    return out

# def run_once_for_question(question: str, seed: int | None = None) -> dict:
#     # Stage 1: choose filter NAMES + enumerated columns object
#     p1 = build_stage1_prompt(question)
#     t1 = call_model_with_retries(p1, seed=seed)
#
#     try:
#         o1 = extract_json(t1)
#     except Exception:
#         o1 = {}
#
#     selected_filter_names = o1.get("selected_filter_names", []) if isinstance(o1, dict) else []
#     selected_column_obj = o1.get("selected_column", {}) if isinstance(o1, dict) else {}
#
#     selected_filter_names = normalize_filter_names(selected_filter_names)
#     selected_filter_names = canonicalize_names(normalize_filter_names(selected_filter_names))
#     selected_column_obj = normalize_selected_column_object(selected_column_obj)
#
#
#
#
#     # Stage 2: pick specific values for those chosen filter NAMES
#     p2 = build_stage2_prompt(question, selected_filter_names)
#     t2 = call_model_with_retries(p2, seed=seed)
#
#     try:
#         o2 = extract_json(t2)
#     except Exception:
#         o2 = {}
#
#     selected_filter = normalize_selected_filter(o2.get("selected_filter", {})) if isinstance(o2, dict) else {}
#
#     # Final merged output EXACTLY as requested by user
#     merged = {
#         "selected_filter": selected_filter,
#         "selected_column": selected_column_obj,
#     }
#     return merged
#
#
# def run_n_times_for_question(question: str, n_runs: int):
#     outputs = []
#     for i in range(n_runs):
#         seed = random.randint(1, 10_000_000)
#         merged = run_once_for_question(question, seed=seed)
#         outputs.append(json.dumps(merged, ensure_ascii=False))
#         time.sleep(DELAY_BETWEEN_CALLS_SEC)
#     return outputs

def run_once_for_question(question: str, seed: int | None = None) -> tuple[dict, dict]:
    # Stage 1
    p1 = build_stage1_prompt(question)
    start1 = time.time()
    t1 = call_model_with_retries(p1, seed=seed)
    elapsed1 = time.time() - start1
    try:
        o1 = extract_json(t1)
    except Exception:
        o1 = {}
    selected_filter_names = normalize_filter_names(o1.get("selected_filter_names", []))
    selected_column_obj = normalize_selected_column_object(o1.get("selected_column", {}))

    # Stage 2
    p2 = build_stage2_prompt(question, selected_filter_names)
    start2 = time.time()
    t2 = call_model_with_retries(p2, seed=seed)
    elapsed2 = time.time() - start2
    try:
        o2 = extract_json(t2)
    except Exception:
        o2 = {}
    selected_filter = normalize_selected_filter(o2.get("selected_filter", {}))

    merged = {
        "selected_filter": selected_filter,
        "selected_column": selected_column_obj,
    }
    timings = {
        "stage1_time_sec": elapsed1,
        "stage2_time_sec": elapsed2,
        "total_time_sec": elapsed1+elapsed2,
    }
    return merged, timings

def run_n_times_for_question(question: str, n_runs: int):
    outputs, timings_list = [], []
    for i in range(n_runs):
        seed = random.randint(1,10_000_000)
        merged, timings = run_once_for_question(question, seed=seed)
        outputs.append(json.dumps(merged, ensure_ascii=False))
        timings_list.append(timings)
        time.sleep(DELAY_BETWEEN_CALLS_SEC)
    return outputs, timings_list


# ============================
# MAIN
# ============================
# --- Canonical names for FILTER CATEGORIES (keys) ---
FILTER_NAME_CANON = {
    # Core from GTs
    "Class of ICI": "Class of ICI",
    "Cancer type": "Cancer type",
    "Monotherapy/combination": "Monotherapy/combination",
    "Trial phase": "Trial phase",
    "Clinical setting in relation to surgery": "Clinical setting in relation to surgery",
    "Primary endpoint": "Primary endpoint",
    "Total sample size": "Total sample size",
    "Year": "Year",

    "ICI Class": "Class of ICI",
    "ICI class": "Class of ICI",
    "Checkpoint class": "Class of ICI",

    "Cancer Type": "Cancer type",
    "Disease": "Cancer type",
    "Indication": "Cancer type",
    "Cancer type – The specific disease/indication under study (e.g., metastatic castration-sensitive prostate cancer, NSCLC, melanoma), including stage/setting when relevant.": "Cancer type",

    "Type of Therapy": "Monotherapy/combination",
    "Monotherapy/Combination": "Monotherapy/combination",
    "Monotherapy vs Combination": "Monotherapy/combination",
    "Mono/Combo": "Monotherapy/combination",

    "Trial Phase": "Trial phase",
    "Clinical Trial Phase": "Trial phase",
    "Phase": "Trial phase",

    "Clinical setting": "Clinical setting in relation to surgery",
    "Perioperative setting": "Clinical setting in relation to surgery",
    "Setting in relation to surgery": "Clinical setting in relation to surgery",
}

# def main():
#     df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)
#
#     # auto-detect question column if needed
#     global QUESTION_COLUMN
#     if QUESTION_COLUMN not in df.columns:
#         candidates = [c for c in df.columns if str(c).strip().lower() in {"question", "query", "prompt"}]
#         if candidates:
#             QUESTION_COLUMN = candidates[0]
#         else:
#             raise ValueError(
#                 f"Couldn't find a question column named '{QUESTION_COLUMN}'. Available columns: {list(df.columns)}"
#             )
#
#     # Prepare output columns: run_1..run_N for merged JSON
#     run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
#     for c in run_cols:
#         if c not in df.columns:
#             df[c] = ""
#
#     # Iterate and fill
#     for idx, row in df.iterrows():
#         q = str(row[QUESTION_COLUMN]).strip()
#         if not q or q.lower() == "nan":
#             continue
#         print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
#         merged_runs = run_n_times_for_question(q, RUNS_PER_QUESTION)
#         for i, merged_json in enumerate(merged_runs):
#             df.at[idx, run_cols[i]] = merged_json
#
#     # Save
#     df.to_excel(EXCEL_PATH_OUT, index=False)
#     print(f"Saved results to: {EXCEL_PATH_OUT}")
#
#
# if __name__ == "__main__":
#     main()

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question","query","prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(f"Couldn't find a question column. Available: {list(df.columns)}")

    run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
    for c in run_cols:
        if c not in df.columns:
            df[c] = ""

    timings_records = []

    for idx,row in df.iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower()=="nan": continue
        print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
        merged_runs, timings_list = run_n_times_for_question(q, RUNS_PER_QUESTION)
        for i,merged_json in enumerate(merged_runs):
            df.at[idx, run_cols[i]] = merged_json
        timings_records.append({
            "row_index": idx,
            "question": q,
            **{f"run_{i+1}_stage1_sec": t["stage1_time_sec"] for i,t in enumerate(timings_list)},
            **{f"run_{i+1}_stage2_sec": t["stage2_time_sec"] for i,t in enumerate(timings_list)},
            **{f"run_{i+1}_total_sec":  t["total_time_sec"]  for i,t in enumerate(timings_list)},
        })

    timings_df = pd.DataFrame(timings_records)
    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT}")

if __name__ == "__main__":
    main()


Processing row 0: Show me the treatment regimen and names of immune checkpoint inhibitors of the c...
Processing row 1: List the names and clinical setting of all combination ICIs given in melanoma pa...
Processing row 2: What are the primary endpoints and sample sizes in trials assessing CTLA4 agents...
Processing row 3: Get all phase 3 studies assessing monotherapy for urothelial/bladder cancer and ...
Processing row 4: Which trials included patients after surgery in NSCLC?
Processing row 5: Provide treatment and comparator names of trials comparing CTLA4 and PD1 agents ...
Processing row 6: Provide all information of studies assessing PD-L1 inhibitors.
Processing row 7: Provide trial names of all the studies where the sample size is equal to or grea...
Processing row 8: Provide treatment names for all the trials where the primary endpoint is overall...
Processing row 9: Provide trial names of all the studies after 2020
Processing row 10: Which cancers have PD-1 inhibitor monotherapy

In [56]:
"""
Run this cell to produce a table to compare ground truth results with the 5 runs from Gemini. FOR PIPELINE 1
"""


import pandas as pd, re, json
from datetime import datetime

# Load the Excel file
file_path = 'runs/query-chosen-filters-columns_with_runs_pipeline1.xlsx'  # adjust if needed
data = pd.read_excel(file_path, sheet_name='results')

# Helper: extract only the JSON object from a messy string (code fences, explanations, etc.)
def extract_json(text):
    if not isinstance(text, str):
        return text
    # Strip code fence markers if present
    cleaned = text.replace("```json", "").replace("```", "").strip()
    # Try to isolate the JSON block
    m = re.search(r'\{[\s\S]*\}', cleaned)
    cleaned = m.group(0).strip() if m else cleaned.strip()
    # Normalize smart quotes
    cleaned = cleaned.replace("“", '"').replace("”", '"').replace("’", "'")
    # Try to pretty-format valid JSON; if it fails, just return the cleaned block
    try:
        return json.dumps(json.loads(cleaned), indent=2, ensure_ascii=False)
    except Exception:
        return cleaned

# Columns to process
json_columns = ['Ground Truth JSON', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5']
json_columns = [c for c in json_columns if c in data.columns]  # handle missing safely

# Extract/clean JSON for each run column
for col in json_columns:
    data[col] = data[col].apply(extract_json)

# Save cleaned version
output_path = 'gt-vs-runs_pipeline1.xlsx'
data.to_excel(output_path, index=False)

print(f"Saved cleaned file as {output_path}")


Saved cleaned file as gt-vs-runs_pipeline1.xlsx


In [57]:
"""
Run this cell to produce a table to compare ground truth results with the 5 runs from Gemini. FOR PIPELINE 2
"""


import pandas as pd, re, json
from datetime import datetime

# Load the Excel file
file_path = 'runs/query-chosen-filters-columns_with_runs_pipeline2.xlsx'  # adjust if needed
data = pd.read_excel(file_path, sheet_name='results')

# Helper: extract only the JSON object from a messy string (code fences, explanations, etc.)
def extract_json(text):
    if not isinstance(text, str):
        return text
    # Strip code fence markers if present
    cleaned = text.replace("```json", "").replace("```", "").strip()
    # Try to isolate the JSON block
    m = re.search(r'\{[\s\S]*\}', cleaned)
    cleaned = m.group(0).strip() if m else cleaned.strip()
    # Normalize smart quotes
    cleaned = cleaned.replace("“", '"').replace("”", '"').replace("’", "'")
    # Try to pretty-format valid JSON; if it fails, just return the cleaned block
    try:
        return json.dumps(json.loads(cleaned), indent=2, ensure_ascii=False)
    except Exception:
        return cleaned

# Columns to process
json_columns = ['Ground Truth JSON', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5']
json_columns = [c for c in json_columns if c in data.columns]  # handle missing safely

# Extract/clean JSON for each run column
for col in json_columns:
    data[col] = data[col].apply(extract_json)

# Save cleaned version
output_path = 'gt-vs-runs_pipeline2.xlsx'
data.to_excel(output_path, index=False)

print(f"Saved cleaned file as {output_path}")


Saved cleaned file as gt-vs-runs_pipeline2.xlsx


In [58]:
!pip install xlsxwriter



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [60]:
"""
This is the cell to run when we wish to get precision and recall values
for filter and column selection (with canonicalization + relaxed scoring).
SCORE FOR PIPELINE 1 SPECIFICALLY.
"""

import pandas as pd, json, re, difflib
from datetime import datetime
from pathlib import Path

# --- Load cleaned JSONs ---
df = pd.read_excel('gt-vs-runs_pipeline1.xlsx')  # must contain: Ground Truth JSON, run_1..run_5
RUN_COLS = [c for c in ['run_1','run_2','run_3','run_4','run_5'] if c in df.columns]
GT_COL = 'Ground Truth JSON'


COL_CONCISE_PATH = Path("definitions_folder/definitions - aim2 - column concise.txt")
CANON_COLS = []
if COL_CONCISE_PATH.exists():
    for line in COL_CONCISE_PATH.read_text(encoding="utf-8").splitlines():
        # split on common dash separators used in the file
        label = re.split(r"\s+[—–-]\s+", line.strip(), maxsplit=1)[0]
        label = label.strip()
        if label and not label.startswith("#"):
            CANON_COLS.append(label)
CANON_COLS_NORM = {label.lower(): label for label in CANON_COLS}


CATEGORY_ALIASES = {
    'ici class': 'ici class',
    'class of ici': 'ici class',
    'cancer type': 'cancer type',
    'monotherapy/combination': 'type of therapy',
    'type of therapy': 'type of therapy',
}

# Value aliases keyed by (canonical_category, normalized_value)
VALUE_ALIASES = {
    ('ici class','pd1'): 'pd1',
    ('ici class','pd-1'): 'pd1',
    ('ici class','pd 1'): 'pd1',
    ('ici class','pd1 inhibitor'): 'pd1',
    ('ici class','pd-l1'): 'pd-l1',
    ('ici class','pd l1'): 'pd-l1',
    ('ici class','pd-l1 inhibitor'): 'pd-l1',
    ('ici class','ctla-4'): 'ctla-4',
    ('ici class','ctla 4'): 'ctla-4',

    ('cancer type','nsclc'): 'non-small cell lung cancer',
    ('cancer type','non small cell lung cancer'): 'non-small cell lung cancer',
    ('cancer type','non-small cell lung cancer'): 'non-small cell lung cancer',
    # add more tumor-type abbrev mappings here if needed

    ('type of therapy','combination'): 'combination therapy',
    ('type of therapy','combo'): 'combination therapy',
    ('type of therapy','combination therapy'): 'combination therapy',
    ('type of therapy','monotherapy'): 'monotherapy',
}

# --- Helpers ---
def norm_text(s):
    if s is None: return ""
    s = str(s)
    s = s.replace("“","\"").replace("”","\"").replace("’","'")
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

def strip_parens(s):
    return re.sub(r'\s*\([^)]*\)\s*', ' ', s).strip()

def extract_json_block(text):
    if not isinstance(text, str): return ""
    t = text.replace("```json","").replace("```","").strip()
    m = re.search(r'\{[\s\S]*\}', t)
    return m.group(0).strip() if m else t

def parse_obj(text):
    if not isinstance(text, str) or not text.strip():
        return {}
    blob = extract_json_block(text)
    try:
        return json.loads(blob)
    except Exception:
        blob2 = re.sub(r',\s*([}\]])', r'\1', blob)
        try:
            return json.loads(blob2)
        except Exception:
            return {}

def canonicalize_filter_pair(k, v):
    k0 = norm_text(k)
    k0 = CATEGORY_ALIASES.get(k0, k0)          # category alias
    v0 = norm_text(strip_parens(v))
    v0 = VALUE_ALIASES.get((k0, v0), v0)       # value alias within category
    # final tidy (common punctuation/dash variants)
    v0 = v0.replace('–','-').replace('—','-')
    return k0, v0

def snap_to_canonical_col(label):
    if not isinstance(label, str): return ""
    raw = label.strip()
    # keep only the part before dash if it looks like "Label — Definition"
    left = re.split(r"\s+[—–-]\s+", raw, maxsplit=1)[0].strip()
    candidate = norm_text(left)

    if candidate in CANON_COLS_NORM:
        return CANON_COLS_NORM[candidate]

    if CANON_COLS:
        best = max(CANON_COLS, key=lambda c: difflib.SequenceMatcher(None, candidate, c.lower()).ratio())
        score = difflib.SequenceMatcher(None, candidate, best.lower()).ratio()
        if score >= 0.92:    # high-confidence snap
            return best
    return left  # fallback to the cleaned left part

def parse_spec(text):
    obj = parse_obj(text)
    filt = obj.get('selected_filter', {}) or {}
    cols = obj.get('selected_column', {}) or {}

    filt_set = set()
    for k, v in filt.items():
        k1, v1 = canonicalize_filter_pair(k, v)
        if k1 and v1 and v1 != "all":
            filt_set.add((k1, v1))

    col_set = set()
    for v in cols.values():
        lab = snap_to_canonical_col(v)
        lab = norm_text(lab)
        if lab:
            col_set.add(lab)

    return filt_set, col_set

def prf_strict(gt_set, pred_set):
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

def similar(a, b):
    ta = set(re.findall(r'\w+', a.lower()))
    tb = set(re.findall(r'\w+', b.lower()))
    if not ta and not tb: return 1.0
    return len(ta & tb) / len(ta | tb)

def prf_relaxed(gt_set, pred_set, partial_credit=0.5, thresh_full=0.9, thresh_partial=0.75):

    # Build maps for pairwise best matches
    gt_unused = set(gt_set)
    pred_unused = set(pred_set)
    exact_tp = len(gt_set & pred_set)

    # remove exacts first
    gt_unused -= (gt_set & pred_set)
    pred_unused -= (gt_set & pred_set)

    partial_tp = 0.0
    # greedy match remaining by similarity
    for p in list(pred_unused):
        best = None; best_sim = 0.0
        for g in gt_unused:
            # Compare key and value separately for filters; single string for columns
            if isinstance(p, tuple):
                sim = 0.5*similar(p[0], g[0]) + 0.5*similar(p[1], g[1])
            else:
                sim = similar(p, g)
            if sim > best_sim:
                best_sim, best = sim, g
        if best is not None:
            if best_sim >= thresh_full:
                exact_tp += 1
                gt_unused.remove(best)
            elif best_sim >= thresh_partial:
                partial_tp += partial_credit
                gt_unused.remove(best)

    tp = exact_tp + partial_tp
    fp = max(0.0, len(pred_set) - (exact_tp + partial_tp))
    fn = max(0.0, len(gt_set) - (exact_tp + partial_tp))
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

# --- Per-row, per-run metrics (filters & columns separately) ---
rows = []
for idx, row in df.iterrows():
    gt_filters, gt_cols = parse_spec(row[GT_COL])
    for r in RUN_COLS:
        run_filters, run_cols = parse_spec(row[r])

        # strict
        _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
        _, _, _, pC, rC = prf_strict(gt_cols, run_cols)

        # relaxed (diagnostic)
        _, _, _, pF_rel, rF_rel = prf_relaxed(gt_filters, run_filters)
        _, _, _, pC_rel, rC_rel = prf_relaxed(gt_cols, run_cols)

        rows.append({
            'row_id': idx,
            'run': r,
            'precision_filters': pF, 'recall_filters': rF,
            'precision_columns': pC, 'recall_columns': rC,
            'precision_filters_relaxed': pF_rel, 'recall_filters_relaxed': rF_rel,
            'precision_columns_relaxed': pC_rel, 'recall_columns_relaxed': rC_rel,
        })

per_run = pd.DataFrame(rows)

# --- Averages across the runs, per query ---
avg_per_query = per_run.groupby('row_id').agg(
    avg_precision_filters=('precision_filters', 'mean'),
    avg_recall_filters=('recall_filters', 'mean'),
    avg_precision_columns=('precision_columns', 'mean'),
    avg_recall_columns=('recall_columns', 'mean'),
    avg_precision_filters_relaxed=('precision_filters_relaxed', 'mean'),
    avg_recall_filters_relaxed=('recall_filters_relaxed', 'mean'),
    avg_precision_columns_relaxed=('precision_columns_relaxed', 'mean'),
    avg_recall_columns_relaxed=('recall_columns_relaxed', 'mean'),
).reset_index()

# --- Merge back to original dataframe so each query has its averages ---
df_with_avgs = df.merge(avg_per_query, left_index=True, right_on='row_id')

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- Save both detailed and summary outputs ---
out_path = f'results/gt-vs-runs-with-filter-column-averages_pipeline1.xlsx'
with pd.ExcelWriter(out_path, engine='xlsxwriter') as xw:
    df_with_avgs.to_excel(xw, sheet_name='queries_with_averages', index=False)
    per_run.to_excel(xw, sheet_name='per_row_per_run', index=False)

print("Saved:", out_path)


Saved: results/gt-vs-runs-with-filter-column-averages_pipeline1.xlsx


In [61]:
"""
This is the cell to run when we wish to get precision and recall values
for filter and column selection (with canonicalization + relaxed scoring).
SCORE FOR PIPELINE 2 SPECIFICALLY.
"""

import pandas as pd, json, re, difflib
from datetime import datetime
from pathlib import Path

# --- Load cleaned JSONs ---
df = pd.read_excel('gt-vs-runs_pipeline2.xlsx')  # must contain: Ground Truth JSON, run_1..run_5
RUN_COLS = [c for c in ['run_1','run_2','run_3','run_4','run_5'] if c in df.columns]
GT_COL = 'Ground Truth JSON'


COL_CONCISE_PATH = Path("definitions_folder/definitions - aim2 - column concise.txt")
CANON_COLS = []
if COL_CONCISE_PATH.exists():
    for line in COL_CONCISE_PATH.read_text(encoding="utf-8").splitlines():
        # split on common dash separators used in the file
        label = re.split(r"\s+[—–-]\s+", line.strip(), maxsplit=1)[0]
        label = label.strip()
        if label and not label.startswith("#"):
            CANON_COLS.append(label)
CANON_COLS_NORM = {label.lower(): label for label in CANON_COLS}


CATEGORY_ALIASES = {
    'ici class': 'ici class',
    'class of ici': 'ici class',
    'cancer type': 'cancer type',
    'monotherapy/combination': 'type of therapy',
    'type of therapy': 'type of therapy',
}

# Value aliases keyed by (canonical_category, normalized_value)
VALUE_ALIASES = {
    ('ici class','pd1'): 'pd1',
    ('ici class','pd-1'): 'pd1',
    ('ici class','pd 1'): 'pd1',
    ('ici class','pd1 inhibitor'): 'pd1',
    ('ici class','pd-l1'): 'pd-l1',
    ('ici class','pd l1'): 'pd-l1',
    ('ici class','pd-l1 inhibitor'): 'pd-l1',
    ('ici class','ctla-4'): 'ctla-4',
    ('ici class','ctla 4'): 'ctla-4',

    ('cancer type','nsclc'): 'non-small cell lung cancer',
    ('cancer type','non small cell lung cancer'): 'non-small cell lung cancer',
    ('cancer type','non-small cell lung cancer'): 'non-small cell lung cancer',
    # add more tumor-type abbrev mappings here if needed

    ('type of therapy','combination'): 'combination therapy',
    ('type of therapy','combo'): 'combination therapy',
    ('type of therapy','combination therapy'): 'combination therapy',
    ('type of therapy','monotherapy'): 'monotherapy',
}

# --- Helpers ---
def norm_text(s):
    if s is None: return ""
    s = str(s)
    s = s.replace("“","\"").replace("”","\"").replace("’","'")
    s = re.sub(r'\s+', ' ', s).strip().lower()
    return s

def strip_parens(s):
    return re.sub(r'\s*\([^)]*\)\s*', ' ', s).strip()

def extract_json_block(text):
    if not isinstance(text, str): return ""
    t = text.replace("```json","").replace("```","").strip()
    m = re.search(r'\{[\s\S]*\}', t)
    return m.group(0).strip() if m else t

def parse_obj(text):
    if not isinstance(text, str) or not text.strip():
        return {}
    blob = extract_json_block(text)
    try:
        return json.loads(blob)
    except Exception:
        blob2 = re.sub(r',\s*([}\]])', r'\1', blob)
        try:
            return json.loads(blob2)
        except Exception:
            return {}

def canonicalize_filter_pair(k, v):
    k0 = norm_text(k)
    k0 = CATEGORY_ALIASES.get(k0, k0)          # category alias
    v0 = norm_text(strip_parens(v))
    v0 = VALUE_ALIASES.get((k0, v0), v0)       # value alias within category
    # final tidy (common punctuation/dash variants)
    v0 = v0.replace('–','-').replace('—','-')
    return k0, v0

def snap_to_canonical_col(label):
    if not isinstance(label, str): return ""
    raw = label.strip()
    # keep only the part before dash if it looks like "Label — Definition"
    left = re.split(r"\s+[—–-]\s+", raw, maxsplit=1)[0].strip()
    candidate = norm_text(left)

    if candidate in CANON_COLS_NORM:
        return CANON_COLS_NORM[candidate]

    if CANON_COLS:
        best = max(CANON_COLS, key=lambda c: difflib.SequenceMatcher(None, candidate, c.lower()).ratio())
        score = difflib.SequenceMatcher(None, candidate, best.lower()).ratio()
        if score >= 0.92:    # high-confidence snap
            return best
    return left  # fallback to the cleaned left part

def parse_spec(text):
    obj = parse_obj(text)
    filt = obj.get('selected_filter', {}) or {}
    cols = obj.get('selected_column', {}) or {}

    filt_set = set()
    for k, v in filt.items():
        k1, v1 = canonicalize_filter_pair(k, v)
        if k1 and v1 and v1 != "all":
            filt_set.add((k1, v1))

    col_set = set()
    for v in cols.values():
        lab = snap_to_canonical_col(v)
        lab = norm_text(lab)
        if lab:
            col_set.add(lab)

    return filt_set, col_set

def prf_strict(gt_set, pred_set):
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

def similar(a, b):
    ta = set(re.findall(r'\w+', a.lower()))
    tb = set(re.findall(r'\w+', b.lower()))
    if not ta and not tb: return 1.0
    return len(ta & tb) / len(ta | tb)

def prf_relaxed(gt_set, pred_set, partial_credit=0.5, thresh_full=0.9, thresh_partial=0.75):

    # Build maps for pairwise best matches
    gt_unused = set(gt_set)
    pred_unused = set(pred_set)
    exact_tp = len(gt_set & pred_set)

    # remove exacts first
    gt_unused -= (gt_set & pred_set)
    pred_unused -= (gt_set & pred_set)

    partial_tp = 0.0
    # greedy match remaining by similarity
    for p in list(pred_unused):
        best = None; best_sim = 0.0
        for g in gt_unused:
            # Compare key and value separately for filters; single string for columns
            if isinstance(p, tuple):
                sim = 0.5*similar(p[0], g[0]) + 0.5*similar(p[1], g[1])
            else:
                sim = similar(p, g)
            if sim > best_sim:
                best_sim, best = sim, g
        if best is not None:
            if best_sim >= thresh_full:
                exact_tp += 1
                gt_unused.remove(best)
            elif best_sim >= thresh_partial:
                partial_tp += partial_credit
                gt_unused.remove(best)

    tp = exact_tp + partial_tp
    fp = max(0.0, len(pred_set) - (exact_tp + partial_tp))
    fn = max(0.0, len(gt_set) - (exact_tp + partial_tp))
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

# --- Per-row, per-run metrics (filters & columns separately) ---
rows = []
for idx, row in df.iterrows():
    gt_filters, gt_cols = parse_spec(row[GT_COL])
    for r in RUN_COLS:
        run_filters, run_cols = parse_spec(row[r])

        # strict
        _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
        _, _, _, pC, rC = prf_strict(gt_cols, run_cols)

        # relaxed (diagnostic)
        _, _, _, pF_rel, rF_rel = prf_relaxed(gt_filters, run_filters)
        _, _, _, pC_rel, rC_rel = prf_relaxed(gt_cols, run_cols)

        rows.append({
            'row_id': idx,
            'run': r,
            'precision_filters': pF, 'recall_filters': rF,
            'precision_columns': pC, 'recall_columns': rC,
            'precision_filters_relaxed': pF_rel, 'recall_filters_relaxed': rF_rel,
            'precision_columns_relaxed': pC_rel, 'recall_columns_relaxed': rC_rel,
        })

per_run = pd.DataFrame(rows)

# --- Averages across the runs, per query ---
avg_per_query = per_run.groupby('row_id').agg(
    avg_precision_filters=('precision_filters', 'mean'),
    avg_recall_filters=('recall_filters', 'mean'),
    avg_precision_columns=('precision_columns', 'mean'),
    avg_recall_columns=('recall_columns', 'mean'),
    avg_precision_filters_relaxed=('precision_filters_relaxed', 'mean'),
    avg_recall_filters_relaxed=('recall_filters_relaxed', 'mean'),
    avg_precision_columns_relaxed=('precision_columns_relaxed', 'mean'),
    avg_recall_columns_relaxed=('recall_columns_relaxed', 'mean'),
).reset_index()

# --- Merge back to original dataframe so each query has its averages ---
df_with_avgs = df.merge(avg_per_query, left_index=True, right_on='row_id')

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- Save both detailed and summary outputs ---
out_path = f'results/gt-vs-runs-with-filter-column-averages_pipeline2.xlsx'
with pd.ExcelWriter(out_path, engine='xlsxwriter') as xw:
    df_with_avgs.to_excel(xw, sheet_name='queries_with_averages', index=False)
    per_run.to_excel(xw, sheet_name='per_row_per_run', index=False)

print("Saved:", out_path)


Saved: results/gt-vs-runs-with-filter-column-averages_pipeline2.xlsx


In [62]:
import pandas as pd
from pathlib import Path
from xlsxwriter.utility import xl_col_to_name  # safer than chr(65+col)

METRICS = [
    "avg_precision_filters",
    "avg_recall_filters",
    "avg_precision_columns",
    "avg_recall_columns",
    "avg_precision_filters_relaxed",
    "avg_recall_filters_relaxed",
    "avg_precision_columns_relaxed",
    "avg_recall_columns_relaxed",
]

def load_queries_with_averages(path: str) -> pd.DataFrame:
    book = pd.read_excel(path, sheet_name=None)
    df = book["queries_with_averages"].copy()
    if "row_id" not in df.columns:
        df = df.reset_index().rename(columns={"index": "row_id"})
    if "Query" not in df.columns:
        df["Query"] = df["row_id"].astype(str)
    return df

def merge_pipelines(p1: pd.DataFrame, p2: pd.DataFrame) -> pd.DataFrame:
    merged = p1[["row_id","Query"]+METRICS].merge(
        p2[["row_id"]+METRICS],
        on="row_id",
        suffixes=("_p1","_p2")
    )
    return merged

def compute_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for m in METRICS:
        m1, m2 = m+"_p1", m+"_p2"
        if m1 not in df or m2 not in df:
            continue
        s1, s2 = df[m1], df[m2]
        rows.append({
            "metric": m,
            "p1_avg": s1.mean(),
            "p2_avg": s2.mean(),
            "p1_wins": int((s1 > s2).sum()),
            "p2_wins": int((s2 > s1).sum()),
            "ties":     int((s1 == s2).sum()),
        })
    return pd.DataFrame(rows)

# NEW: explicit green/yellow/red counts per column (mirrors your CF rules)
def compute_color_counts(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each metric, returns counts of green/yellow/red for p1 column and p2 column.
    Green in a column == that pipeline's value > the other pipeline's value.
    Yellow == tie. Red == that pipeline's value < the other's.
    """
    rows = []
    for m in METRICS:
        m1, m2 = m+"_p1", m+"_p2"
        if m1 not in df or m2 not in df:
            continue
        s1, s2 = df[m1], df[m2]

        p1_green = int((s1 > s2).sum())
        p1_yellow = int((s1 == s2).sum())
        p1_red = int((s1 < s2).sum())

        p2_green = int((s2 > s1).sum())
        p2_yellow = p1_yellow
        p2_red = p1_green  # when p1 is green, p2 is red, and vice versa

        rows.append({
            "metric": m,
            # counts for the p1 column colors
            "p1_green": p1_green,
            "p1_yellow": p1_yellow,
            "p1_red": p1_red,
            # counts for the p2 column colors
            "p2_green": p2_green,
            "p2_yellow": p2_yellow,
            "p2_red": p2_red,
        })
    return pd.DataFrame(rows)

def write_with_formatting(df: pd.DataFrame, summary: pd.DataFrame, out_path: str):
    color_counts = compute_color_counts(df)

    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="query_by_query", index=False)
        summary.to_excel(writer, sheet_name="summary", index=False)
        color_counts.to_excel(writer, sheet_name="color_counts", index=False)

        wb = writer.book
        ws = writer.sheets["query_by_query"]

        fmt_p1 = wb.add_format({"bg_color":"#C6EFCE"})  # green
        fmt_p2 = wb.add_format({"bg_color":"#FFC7CE"})  # red
        fmt_tie = wb.add_format({"bg_color":"#FFEB9C"}) # yellow

        ws.freeze_panes(1, 0)
        n_rows = len(df) + 1  # +1 for header

        for col_i, col in enumerate(df.columns):
            if not col.endswith("_p1"):
                continue
            base = col[:-3]
            col_p2 = base + "_p2"
            if col_p2 not in df:
                continue

            # safe column letters (A, B, ..., AA, AB, ...)
            c1 = xl_col_to_name(col_i)        # p1 column letter(s)
            c2 = xl_col_to_name(df.columns.get_loc(col_p2))

            rng1 = f"{c1}2:{c1}{n_rows}"  # data rows only
            rng2 = f"{c2}2:{c2}{n_rows}"

            # format rules for P1 col (green if p1>p2, red if p1<p2, yellow if tie)
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2>{c2}2", "format":fmt_p1})
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2<{c2}2", "format":fmt_p2})
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2={c2}2", "format":fmt_tie})

            # format rules for P2 col (green if p2>p1, red if p2<p1, yellow if tie)
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2>{c1}2", "format":fmt_p1})
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2<{c1}2", "format":fmt_p2})
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2={c1}2", "format":fmt_tie})

# === Example usage inside notebook ===
p1_path = "results/gt-vs-runs-with-filter-column-averages_pipeline1.xlsx"
p2_path = "results/gt-vs-runs-with-filter-column-averages_pipeline2.xlsx"
out_path = "results/pipeline_comparison.xlsx"

df1 = load_queries_with_averages(p1_path)
df2 = load_queries_with_averages(p2_path)

merged = merge_pipelines(df1, df2)
summary = compute_summary(merged)

write_with_formatting(merged, summary, out_path)
print("Comparison written to:", out_path)


Comparison written to: results/pipeline_comparison.xlsx
